In [74]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [7]:
loader=PyPDFLoader("Anthropogenic_disturbance.pdf")
data=loader.load()


In [14]:
print(data[0].page_content)

Anthropogenic disturbance expands the climatic
limits of annual plant dominance
July 10, 2026
Keywords:climate change; functional groups; functional traits; grasslands; life cy-
cle; life-history; lifespan; perennials; rainfall; Disturbance and Recovery Across Global
Grasslands Network (DRAGNet)
Abstract
Disturbance regimes and nutrient inputs are changing worldwide, with con-
sequences for the structure and functioning of plant communities. Classical life-
history theory predicts that disturbance should shift communities from long-lived
perennials toward short-lived annuals, and that nutrient enrichment may amplify
this shift. However, these predictions have not been tested experimentally across
broad environmental gradients. Here, using a global coordinated grassland exper-
iment spanning 37 sites, we tested how physical disturbance (vegetation removal
and shallow soil tillage) and fertilisation reshape annual–perennial balance, and
whether disturbance relaxes the climatic limits of 

In [22]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs=splitter.split_documents(data)

In [23]:
print(docs[0])

page_content='Anthropogenic disturbance expands the climatic
limits of annual plant dominance
July 10, 2026
Keywords:climate change; functional groups; functional traits; grasslands; life cy-
cle; life-history; lifespan; perennials; rainfall; Disturbance and Recovery Across Global
Grasslands Network (DRAGNet)
Abstract
Disturbance regimes and nutrient inputs are changing worldwide, with con-
sequences for the structure and functioning of plant communities. Classical life-
history theory predicts that disturbance should shift communities from long-lived
perennials toward short-lived annuals, and that nutrient enrichment may amplify
this shift. However, these predictions have not been tested experimentally across
broad environmental gradients. Here, using a global coordinated grassland exper-
iment spanning 37 sites, we tested how physical disturbance (vegetation removal
and shallow soil tillage) and fertilisation reshape annual–perennial balance, and' metadata={'producer': 'pikepdf 8.15.

In [27]:
embedding = HuggingFaceEndpointEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2',
                                           task='feature-extraction')

In [28]:
vectorstore = Chroma(
    collection_name="anthropogenic_disturbance",
    embedding_function=embedding,
    persist_directory="./chroma_db"
)

In [29]:
vectorstore.add_documents(docs)

<bound method VectorStore.add_documents of <langchain_chroma.vectorstores.Chroma object at 0x0000023004021950>>

In [70]:
def input_question(x):
    question=(x)
    return question

input_question_runnable = RunnableLambda(input_question)


In [82]:
# prompt_question=PromptTemplate(
#     template="""
#     {question}""",

#     inputvariables=["question"]

# )
prompt_final=PromptTemplate(
    template="""You are a helpful assistant. Use the following context to answer the question at the end. 
    If you don't know the answer, just say you don't know, don't try to make up an answer.
    Here is the context: 
    {context}
    Now answer the question with the help of the context provided. If the context is not sufficient to answer the question, say "I don't know".
    Question: {question}
    """,
    inputvariables=['context','question']
)

In [36]:
load_dotenv()

True

In [39]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)
model = ChatHuggingFace(
llm=llm,
)

In [41]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":4, 
                   "fetch_k":10,}
)

In [47]:
parser=StrOutputParser()

In [75]:
passchain=RunnablePassthrough(
    question=RunnablePassthrough(),
)

In [ ]:
# chain = input_question_runnable | prompt_question | retriever | prompt_final | model

parallel_chain = RunnableParallel(
    question=passchain,
    context=retriever
)
final_chain= input_question_runnable | parallel_chain | prompt_final | model


In [81]:
result=final_chain.invoke(  {"question": "what is anthropogenic disturbance?"})

AttributeError: 'StringPromptValue' object has no attribute 'replace'